In [1]:
import numpy as np
import pandas as pd


In [2]:
df=pd.read_csv("/home/ammarkhalid/Machine Learning/100 days of Machine Learning/Column Transformer/Datasets/covid_toy.csv")

In [3]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [11]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    str    
 2   fever      90 non-null     float64
 3   cough      100 non-null    str    
 4   city       100 non-null    str    
 5   has_covid  100 non-null    str    
dtypes: float64(1), int64(1), str(4)
memory usage: 6.7 KB


### Without Column Transformer

In [5]:
from sklearn.model_selection import train_test_split

x_train,x_test,t_train,y_test=train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.3)


In [6]:
#imputing fever
from sklearn.impute import SimpleImputer
si=SimpleImputer()

x_train_fever=si.fit_transform(x_train[['fever']])
x_test_fever=si.transform(x_test[['fever']])


In [13]:
x_train_fever.shape,x_test_fever.shape

((70, 1), (30, 1))

In [12]:
# Ordinal Encoding
from sklearn.preprocessing import OrdinalEncoder
oe=OrdinalEncoder(categories=[['Mild','Strong']])
x_train_cough=oe.fit_transform(x_train[['cough']])
x_test_cough=oe.transform(x_test[['cough']])

In [14]:
x_train_cough.shape,x_test_cough.shape

((70, 1), (30, 1))

In [17]:
#OneHot Encoding
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(drop='first')
x_train_gender_city=ohe.fit_transform(x_train[['gender','city']]).toarray()
x_test_gender_city=ohe.transform(x_test[['gender','city']]).toarray()

In [18]:
x_test_gender_city.shape,x_test_gender_city.shape

((30, 4), (30, 4))

In [35]:
# Extracting Age
x_train_age = x_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
x_test_age = x_test.drop(columns=['gender','fever','cough','city']).values

x_train_age.shape,x_test_age.shape

((70, 1), (30, 1))

In [37]:
final_df=np.concatenate((x_train_age,x_train_cough,x_train_gender_city,x_train_fever),axis=1)
final_df.shape

(70, 7)

In [39]:
x_test_transformed = np.concatenate((x_test_age,x_test_fever,x_test_gender_city,x_test_cough),axis=1)
x_test_transformed.shape

(30, 7)

### With column transformer

In [47]:
from sklearn.compose import ColumnTransformer
transformer=ColumnTransformer(transformers=[
    ('tf1',SimpleImputer(),['fever']),
    ('tf2',OneHotEncoder(drop='first'),['gender','city']),
    ('tf3',OrdinalEncoder(categories=[['Mild','Strong']]),['cough'])],remainder='passthrough')  # by default remainder=drop

x_train_new=transformer.fit_transform(x_train)
x_test_new=transformer.transform(x_test)


In [48]:
x_train_new.shape,x_test_new.shape

((70, 7), (30, 7))